# `ImuFactor` vs. `CombinedImuFactor` Showdown
This notebook demonstrates the equivalence between two ways of modeling IMU measurements in GTSAM:

1.  **`gtsam.ImuFactor`**: This factor relates the poses and velocities at two time steps (`i` and `j`), based on preintegrated IMU measurements. The preintegration itself depends on the bias at time `i`. The evolution of the bias from `i` to `j` is modeled separately using a `gtsam.BetweenFactorConstantBias`.
2.  **`gtsam.CombinedImuFactor`**: This is a more recent addition to GTSAM that combines the functionality of the `ImuFactor` and the `BetweenFactorConstantBias` into a single factor. It directly models the relationship between the full states (pose, velocity, bias) at two consecutive time steps.

Our goal is to show that these two methods produce identical results. We will build two factor graphs in parallel and compare them at each step.

### Step 1: Common Setup and Imports

In [ ]:
# Install gtsam-develop if not installed
try:
    import gtsam
except ImportError:
    %pip install gtsam-develop

In [ ]:
# Imports
import gtsam
import numpy as np
from gtsam.symbol_shorthand import X, V, B

In [ ]:
def compare_matrices_side_by_side(mat1, mat2, precision=4):
  mat1 = np.array(mat1)
  mat2 = np.array(mat2)
  rows = max(mat1.shape[0], mat2.shape[0])
  mat1_str = np.array2string(mat1, precision=precision, suppress_small=True, max_line_width=80).splitlines()
  mat2_str = np.array2string(mat2, precision=precision, suppress_small=True, max_line_width=80).splitlines()
  for i in range(rows):
    left = mat1_str[i] if i < len(mat1_str) else ""
    right = mat2_str[i] if i < len(mat2_str) else ""
    print(f"{left:<50} | {right}")

First, we define the scenario. We will simulate a stationary IMU, so the only acceleration measured should be due to gravity, and there should be no rotation.

In [ ]:
# Scenario Parameters
dt = 0.1  # 10 measurements between states
integration_time = 1.0 # Each factor integrates over 1s (10*dt)
gravity = np.array([0, 0, -9.81])

# IMU measurements (zero motion)
measured_acc = -gravity  # Correct for gravity to get body-frame acceleration
measured_gyro = np.array([0, 0, 0])

Next, we set up the parameters for IMU preintegration. The key difference is that `PreintegrationCombinedParams` includes parameters for the bias random walk (`biasAccCovariance` and `biasOmegaCovariance`), whereas for the standard `ImuFactor`, these are defined later in the `BetweenFactor`'s noise model.

In [ ]:
# Define IMU noise and bias parameters
accel_noise_sigma = 1e-4
gyro_noise_sigma = 1e-4
integration_noise_sigma = 1e-5
bias_acc_walk_sigma = 1e-4
bias_gyro_walk_sigma = 1e-4

# Standard ImuFactor parameters
params_imu = gtsam.PreintegrationParams(gravity)
params_imu.setAccelerometerCovariance(np.eye(3) * accel_noise_sigma**2)
params_imu.setGyroscopeCovariance(np.eye(3) * gyro_noise_sigma**2)
params_imu.setIntegrationCovariance(np.eye(3) * integration_noise_sigma**2)

# CombinedImuFactor parameters
params_combined = gtsam.PreintegrationCombinedParams(gravity)
params_combined.setAccelerometerCovariance(np.eye(3) * accel_noise_sigma**2)
params_combined.setGyroscopeCovariance(np.eye(3) * gyro_noise_sigma**2)
params_combined.setIntegrationCovariance(np.eye(3) * integration_noise_sigma**2)
params_combined.setBiasAccCovariance(np.eye(3) * bias_acc_walk_sigma**2) # Bias random walk
params_combined.setBiasOmegaCovariance(np.eye(3) * bias_gyro_walk_sigma**2) # Bias random walk

bias_prior_sigma = 0.01  # Standard deviation for bias prior
params_combined.setBiasAccOmegaInit(np.eye(6) * bias_prior_sigma**2) # Initial covariance
# params_combined.setBiasAccOmegaInit(np.zeros(6)) # Initial covariance

### Step 2: Preintegration and Covariance Comparison
Now we create the preintegrated measurement objects (`PIM`s) and integrate our simulated measurements. We can then directly compare their preintegrated measurement covariances.

In [ ]:
# We start with a zero-bias assumption for preintegration
bias_0 = gtsam.imuBias.ConstantBias()

# Create the two PIM objects
pim_imu = gtsam.PreintegratedImuMeasurements(params_imu, bias_0)
pim_combined = gtsam.PreintegratedCombinedMeasurements(params_combined, bias_0)

# Integrate measurements
num_steps = int(integration_time / dt)
for _ in range(num_steps):
    pim_imu.integrateMeasurement(measured_acc, measured_gyro, dt)
    pim_combined.integrateMeasurement(measured_acc, measured_gyro, dt)

The standard `PIM` has a 9x9 covariance matrix for the preintegrated rotation, position, and velocity. The `Combined PIM` has a 15x15 covariance, which also includes the uncertainty in the bias evolution. The top-left 9x9 block of the combined covariance should be identical to the standard PIM's covariance.

In [ ]:
cov_imu = pim_imu.preintMeasCov()
cov_combined = pim_combined.preintMeasCov()

print('IMU Factor PIM Covariance (9x9) vs Combined IMU Factor PIM Covariance (15x15):')
compare_matrices_side_by_side(1e9*cov_imu, 1e9*cov_combined, precision=0)

### Step 3: Building Equivalent Factor Graphs
Next, we construct the two factor graphs for a trajectory with two IMU integration steps (three states at times k=0, 1, 2). We will add identical priors to both graphs.

In [ ]:
# Initial state at t=0
pose_0 = gtsam.Pose3()
vel_0 = np.zeros(3)
# The bias_0 variable is the same as the one used for pre-integration above.

# Define noise models for priors
pose_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([0.01]*3 + [0.01]*3))
velocity_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([0.01]*3))
bias_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([bias_prior_sigma]*6))

# The bias random walk noise is defined once for the BetweenFactor
bias_walk_noise_model = gtsam.noiseModel.Diagonal.Sigmas(
    np.concatenate([np.full(3, bias_acc_walk_sigma), np.full(3, bias_gyro_walk_sigma)]) * np.sqrt(integration_time)
)

In [ ]:
graph_imu = gtsam.NonlinearFactorGraph()
graph_combined = gtsam.NonlinearFactorGraph()

# Add identical priors to both graphs
graph_imu.add(gtsam.PriorFactorPose3(X(0), pose_0, pose_noise))
graph_imu.add(gtsam.PriorFactorVector(V(0), vel_0, velocity_noise))
graph_imu.add(gtsam.PriorFactorConstantBias(B(0), bias_0, bias_noise))

graph_combined.add(gtsam.PriorFactorPose3(X(0), pose_0, pose_noise))
graph_combined.add(gtsam.PriorFactorVector(V(0), vel_0, velocity_noise))
graph_combined.add(gtsam.PriorFactorConstantBias(B(0), bias_0, bias_noise))

# Add factors for two steps (0->1 and 1->2)
for k in range(2):
    # ImuFactor graph: Add ImuFactor and BetweenFactor separately
    graph_imu.add(gtsam.ImuFactor(X(k), V(k), X(k + 1), V(k + 1), B(k), pim_imu))
    graph_imu.add(gtsam.BetweenFactorConstantBias(B(k), B(k + 1), gtsam.imuBias.ConstantBias(), bias_walk_noise_model))
    
    # CombinedImuFactor graph: Add a single combined factor
    graph_combined.add(gtsam.CombinedImuFactor(X(k), V(k), X(k+1), V(k+1), B(k), B(k+1), pim_combined))

print('Graph 1 (ImuFactor + BetweenFactor):')
graph_imu.print('')
print('\nGraph 2 (CombinedImuFactor):')
graph_combined.print('')

### Step 4: Creating Initial Estimates
We need to provide an initial guess for the optimizer. Since the problem is simple (no motion), we can initialize all states and biases to zero. The initial estimates for both graphs will be identical.

In [ ]:
initial_estimate = gtsam.Values()

for k in range(3):
    initial_estimate.insert(X(k), pose_0)
    initial_estimate.insert(V(k), vel_0)
    initial_estimate.insert(B(k), bias_0)

print('Initial Estimate:\n', initial_estimate)

### Step 5: Comparing Factor Jacobians
Before optimizing, we can inspect the factors themselves. A nonlinear factor's `linearize` method produces a `JacobianFactor`, which is a linear approximation of the factor's error function around a given state. However, it does mix in the PIM covariances, which we do know are different from Step 2. Hence, here we compare the linearization using numerical derivatives of `evaluateError`, which gives us the *unwhitened* error. 

Below we show that the combination of Jacobians from `ImuFactor` and `BetweenFactor` for the interval k=0 to k=1 is identical to the single Jacobian from `CombinedImuFactor`, at least for the first 9 rows, which correspond to the NavState error.

In [ ]:
imu_factor_01 = gtsam.ImuFactor(X(0), V(0), X(1), V(1), B(0), pim_imu)
combined_factor_01 = gtsam.CombinedImuFactor(X(0), V(0), X(1), V(1), B(0), B(1), pim_combined)

For the first four arguments (really NavState 0 and NavState 1) the Jacobians are indeed identical:

In [ ]:
from gtsam.utils.numerical_derivative import numericalDerivative51, numericalDerivative61
print('Comparing Jacobians w.r.t. pose:')
H1 = numericalDerivative51(imu_factor_01.evaluateError, pose_0, vel_0, pose_0, vel_0, bias_0)
H2 = numericalDerivative61(combined_factor_01.evaluateError, pose_0, vel_0, pose_0, vel_0, bias_0, bias_0)
compare_matrices_side_by_side(H1, H2, precision=3)

In [ ]:
from gtsam.utils.numerical_derivative import numericalDerivative52, numericalDerivative62
print('Comparing Jacobians w.r.t. vel:')
H1 = numericalDerivative52(imu_factor_01.evaluateError, pose_0, vel_0, pose_0, vel_0, bias_0)
H2 = numericalDerivative62(combined_factor_01.evaluateError, pose_0, vel_0, pose_0, vel_0, bias_0, bias_0)
compare_matrices_side_by_side(H1, H2, precision=3)

In [ ]:
from gtsam.utils.numerical_derivative import numericalDerivative53, numericalDerivative63
print('Comparing Jacobians w.r.t. second pose:')
H1 = numericalDerivative53(imu_factor_01.evaluateError, pose_0, vel_0, pose_0, vel_0, bias_0)
H2 = numericalDerivative63(combined_factor_01.evaluateError, pose_0, vel_0, pose_0, vel_0, bias_0, bias_0)
compare_matrices_side_by_side(H1, H2, precision=3)

In [ ]:
from gtsam.utils.numerical_derivative import numericalDerivative54, numericalDerivative64
print('Comparing Jacobians w.r.t. second velocity:')
H1 = numericalDerivative54(imu_factor_01.evaluateError, pose_0, vel_0, pose_0, vel_0, bias_0)
H2 = numericalDerivative64(combined_factor_01.evaluateError, pose_0, vel_0, pose_0, vel_0, bias_0, bias_0)
compare_matrices_side_by_side(H1, H2, precision=3)

For the 5th argument, the initial bias, we see that again the first 9 rows are identical, but obviously the 6 last rows correspond to the bias evolution, and its Jacobian is $I_6$ in the first bias.

In [ ]:
from gtsam.utils.numerical_derivative import numericalDerivative55, numericalDerivative65
print('Comparing Jacobians w.r.t. bias B(0):')
H1 = numericalDerivative55(imu_factor_01.evaluateError, pose_0, vel_0, pose_0, vel_0, bias_0)
H2 = numericalDerivative65(combined_factor_01.evaluateError, pose_0, vel_0, pose_0, vel_0, bias_0, bias_0)
compare_matrices_side_by_side(H1, H2, precision=3)

The derivative in B(1) is negative $I_6$:

In [ ]:
from gtsam.utils.numerical_derivative import numericalDerivative66
H2 = numericalDerivative66(combined_factor_01.evaluateError, pose_0, vel_0, pose_0, vel_0, bias_0, bias_0)
print(f'Note: The IMU factor does not have a B(1) term, so we only show the combined factor:\n{H2}')

### Step 6: Optimizing and Comparing Final Results
Now that we've established the underlying mathematical equivalence of the factors, let's optimize both graphs and compare the final results. They should be identical.

In [ ]:
# Optimize both graphs
optimizer_imu = gtsam.LevenbergMarquardtOptimizer(graph_imu, initial_estimate)
result_imu = optimizer_imu.optimize()

optimizer_combined = gtsam.LevenbergMarquardtOptimizer(graph_combined, initial_estimate)
result_combined = optimizer_combined.optimize()

print('--- Results from ImuFactor + BetweenFactor Graph ---')
result_imu.print('')
print('\n--- Results from CombinedImuFactor Graph ---')
result_combined.print('')

# Verify the results are equal
assert(result_imu.equals(result_combined, 1e-6))
print('\nSuccess! The final results from both optimizers are equal.')

Finally, we can check the posterior marginal covariances on the bias estimates. As expected, these are also identical.

In [ ]:
# Analyze marginal covariances
marginals_imu = gtsam.Marginals(graph_imu, result_imu)
marginals_combined = gtsam.Marginals(graph_combined, result_combined)

for k in range(3):
    cov_imu_k = marginals_imu.marginalCovariance(B(k))
    cov_combined_k = marginals_combined.marginalCovariance(B(k))
    
    print(f'\nCovariance for Bias B({k}):')
    compare_matrices_side_by_side(1e6*cov_imu_k, 1e6*cov_combined_k, precision=3)

In [ ]:
# Compare marginal covariances for poses and velocities between both graphs
for k in range(3):
  cov_pose_imu = marginals_imu.marginalCovariance(X(k))
  cov_pose_combined = marginals_combined.marginalCovariance(X(k))
  print(f'\nCovariance for Pose X({k}):')
  compare_matrices_side_by_side(1e6*cov_pose_imu, 1e6*cov_pose_combined, precision=3)

  cov_vel_imu = marginals_imu.marginalCovariance(V(k))
  cov_vel_combined = marginals_combined.marginalCovariance(V(k))
  print(f'\nCovariance for Velocity V({k}):')
  compare_matrices_side_by_side(1e6*cov_vel_imu, 1e6*cov_vel_combined, precision=3)

### Conclusion
We have successfully demonstrated that `gtsam.CombinedImuFactor` is a mathematically equivalent but more convenient way to model IMU measurements compared to using a separate `gtsam.ImuFactor` and `gtsam.BetweenFactorConstantBias`. We showed this by verifying that:
1. Their preintegrated measurement covariances are consistent.
2. The linearization (Jacobians) of a `CombinedImuFactor` is identical to the combined Jacobians of an `ImuFactor` and a `BetweenFactor`.
3. The final optimized results and posterior covariances from factor graphs built with both methods are identical.